### Recommendations using surprise library models:

This notebook uses the surprise library to build a model using surprise library and generate ratings for the test data. \\
**Models:** SVD, Baseline estimation, item-item based KNN collaborative filtering etc are explored. Hyperparameter tuning for the models is performed. \\
**Evaluation:** The test data ratings are stored for further evaluation such as RMSE and MAE. The recommendations are evaluated using precision@5, recall@5, NDCG and overall accuracy. 

In [12]:
from surprise import SVD, BaselineOnly, SVDpp, NMF, SlopeOne, CoClustering, Reader
from surprise import Dataset
from surprise.model_selection import cross_validate
from surprise.prediction_algorithms import KNNBaseline, KNNBasic, KNNWithMeans, KNNWithZScore
from surprise import accuracy
from surprise.model_selection import train_test_split

In [13]:
import math
from collections import defaultdict
import csv
from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd
import time

In [14]:
def convert_traintest_dataframe_forsurprise(training_dataframe, testing_dataframe):
    reader = Reader(rating_scale=(0, 5))
    trainset = Dataset.load_from_df(training_dataframe[['userId', 'movieId', 'rating']], reader)
    testset = Dataset.load_from_df(testing_dataframe[['userId', 'movieId', 'rating']], reader)
    trainset = trainset.construct_trainset(trainset.raw_ratings)
    testset = testset.construct_testset(testset.raw_ratings)
    return trainset, testset

In [15]:
file_path_train = 'training_data.csv'
file_path_test = 'testing_data.csv'
traindf = pd.read_csv(file_path_train)
testdf = pd.read_csv(file_path_test)
trainset, testset = convert_traintest_dataframe_forsurprise(traindf, testdf)

In [16]:
def get_top_n(predictions, n):
    # First map the predictions to each user.
    top_n = defaultdict(list)
    org_ratings = defaultdict(list)

    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
        org_ratings[uid].append((iid, true_r))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n, org_ratings

In [17]:
def dcg_at_k(scores):
    return scores[0] + sum(sc/math.log(ind, 2) for sc, ind in zip(scores[1:], range(2, len(scores) + 1)))

def ndcg_at_k(scores):
    idcg = dcg_at_k(sorted(scores, reverse=True))
    return (dcg_at_k(scores)/idcg) if idcg > 0.0 else 0.0

In [18]:
def precision_recall_at_k(predictions, k=5, threshold=3.5):
    '''Return precision and recall at k metrics for each user.'''

    # First map the predictions to each user.
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():

        # Sort user ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Number of recommended items in top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

        # Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

        # Precision@K: Proportion of recommended items that are relevant
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 1

        # Recall@K: Proportion of relevant items that are recommended
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 1

    precision = (sum(prec for prec in precisions.values()) / len(precisions))
    recall = (sum(rec for rec in recalls.values()) / len(recalls))

    return precision, recall

In [19]:
def recommendation(algo, trainset, testset):
  # Train the algorithm on the trainset, and predict ratings for the testset
  start_fit = time.time()
  algo.fit(trainset)
  end_fit = time.time()
  fit_time = end_fit - start_fit

  # Predictions on testing set
  start_test = time.time()
  test_predictions = algo.test(testset)
  end_test = time.time()
  test_time = end_test - start_test

  test_rmse = accuracy.rmse(test_predictions)
  test_mae = accuracy.mae(test_predictions)

  top_n, org_ratings = get_top_n(test_predictions, 5)

  precision, recall = precision_recall_at_k(test_predictions)

  f_measure = (2*precision*recall)/(precision+recall)

  ndcg_scores = dict()
  for uid, user_ratings in top_n.items():
    scores = []
    for iid, est_r in user_ratings:
        iid_found = False
        org_user_ratings = org_ratings[uid]
        for i, r in org_user_ratings:
            if iid == i:
                scores.append(r)
                iid_found = True
                break
        if not iid_found:
            scores.append(0)
    ndcg_scores[uid] = ndcg_at_k(scores)
  ndcg_score = sum(ndcg for ndcg in ndcg_scores.values())/len(ndcg_scores)

  return (test_rmse, test_mae, fit_time, test_time, precision, recall, f_measure, ndcg_score,test_predictions)

#### Basic algorithm (Baseline approach):

In [21]:
# basic collaborative filtering algorithm taking into account a baseline rating.
sim_options = {'name': 'pearson_baseline',
               'user_based': False  # compute  similarities between items
               }
algo = KNNBaseline(sim_options=sim_options)

results = recommendation(algo,trainset,testset)
print(results[0])
print(results[1])
print(results[2])
print(results[3])
print(results[4])
print(results[5])
print(results[6])
print(results[7])

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
RMSE: 0.8511
MAE:  0.6493
0.8511004357970081
0.649309824895583
2.2836170196533203
1.7609691619873047
0.8410382513661224
0.4139894083098312
0.5548577760252646
0.9645258587912089


In [22]:
algo = CoClustering(2,5,50)

test_rmse, test_mae, test_predictions, fit_time, test_time, precision, recall, f_measure, ndcg_score = recommendation(algo,trainset,testset)
print(test_rmse)
print(test_mae)
print(fit_time)
print(test_time)
print(precision)
print(recall)
print(f_measure)
print(ndcg_score)

RMSE: 0.9356
MAE:  0.7250
0.9355563736646445
0.7250456407450602
0.024937868118286133
0.7928415300546479
0.3905565110510429
0.5233225187785462
0.9562652098686116
[Prediction(uid=1, iid=157, r_ui=5.0, est=3.3675140668177783, details={'was_impossible': False}), Prediction(uid=1, iid=231, r_ui=5.0, est=3.8856864448032398, details={'was_impossible': False}), Prediction(uid=1, iid=457, r_ui=5.0, est=4.834462027436624, details={'was_impossible': False}), Prediction(uid=1, iid=590, r_ui=4.0, est=4.800925273501689, details={'was_impossible': False}), Prediction(uid=1, iid=593, r_ui=4.0, est=4.9957968951006055, details={'was_impossible': False}), Prediction(uid=1, iid=608, r_ui=5.0, est=4.923069622373333, details={'was_impossible': False}), Prediction(uid=1, iid=673, r_ui=3.0, est=3.6303326838429344, details={'was_impossible': False}), Prediction(uid=1, iid=780, r_ui=3.0, est=4.346897121360329, details={'was_impossible': False}), Prediction(uid=1, iid=804, r_ui=4.0, est=5, details={'was_impossib

In [23]:
surprise_df = pd.DataFrame(columns= ['Algorithm', 'test_rmse', 'test_mae', 'fit_time', 'test_time', 'Precision', 'Recall', 'F-measure', 'NDCG'])

In [24]:
# Iterate over all algorithms
for algorithm in [KNNBasic(), SVD(), SVDpp(), SlopeOne(), NMF(), KNNBaseline(), KNNWithMeans(), KNNWithZScore(), BaselineOnly(), CoClustering()]:
    results = recommendation(algorithm,trainset,testset) 
    
    name =str(algorithm).split(' ')[0].split('.')[-1]
    print("Algorithm:", name)
    df = pd.DataFrame([[name, results[0], results[1], results[2], results[3], results[4], results[5], results[6], results[7]]], columns= ['Algorithm', 'test_rmse', 'test_mae', 'fit_time', 'test_time', 'Precision', 'Recall', 'F-measure', 'NDCG'])
    surprise_df = pd.concat([df, surprise_df], ignore_index=True)
surprise_df.sort_values(by='test_rmse', ascending=False) 

Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.9459
MAE:  0.7262
Algorithm: KNNBasic


/var/folders/l_/0gl5f9751hb9q5gc_n_s_dfm0000gn/T/ipykernel_75889/3842485799.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  surprise_df = pd.concat([df, surprise_df], ignore_index=True)


RMSE: 0.8743
MAE:  0.6710
Algorithm: SVD
RMSE: 0.8634
MAE:  0.6619
Algorithm: SVDpp
RMSE: 0.8976
MAE:  0.6877
Algorithm: SlopeOne
RMSE: 0.9188
MAE:  0.7029
Algorithm: NMF
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.8706
MAE:  0.6669
Algorithm: KNNBaseline
Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.8915
MAE:  0.6833
Algorithm: KNNWithMeans
Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.8902
MAE:  0.6772
Algorithm: KNNWithZScore
Estimating biases using als...
RMSE: 0.8702
MAE:  0.6716
Algorithm: BaselineOnly
RMSE: 0.9405
MAE:  0.7294
Algorithm: CoClustering


,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
9,KNNBasic,0.945949,0.726180,0.031668,0.348843,0.790164,0.428001,0.555247,0.961261
0,CoClustering,0.940477,0.729357,0.443287,0.024041,0.792240,0.385911,0.519007,0.954984
5,NMF,0.918784,0.702863,0.483368,0.024606,0.786721,0.393843,0.524909,0.955988
6,SlopeOne,0.897614,0.687702,1.215158,1.542752,0.808579,0.406097,0.540657,0.959408
3,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573
2,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
8,SVD,0.874252,0.670995,0.277401,0.030175,0.820219,0.402056,0.539607,0.960079
4,KNNBaseline,0.870594,0.666896,0.052759,0.494724,0.806011,0.416210,0.548951,0.959268
1,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
7,SVDpp,0.863433,0.661943,18.381992,3.228610,0.832268,0.398290,0.538754,0.962390


In [25]:
surprise_df.sort_values(by='test_rmse') 

,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
7,SVDpp,0.863433,0.661943,18.381992,3.228610,0.832268,0.398290,0.538754,0.962390
1,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
4,KNNBaseline,0.870594,0.666896,0.052759,0.494724,0.806011,0.416210,0.548951,0.959268
8,SVD,0.874252,0.670995,0.277401,0.030175,0.820219,0.402056,0.539607,0.960079
2,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
3,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573
6,SlopeOne,0.897614,0.687702,1.215158,1.542752,0.808579,0.406097,0.540657,0.959408
5,NMF,0.918784,0.702863,0.483368,0.024606,0.786721,0.393843,0.524909,0.955988
0,CoClustering,0.940477,0.729357,0.443287,0.024041,0.792240,0.385911,0.519007,0.954984
9,KNNBasic,0.945949,0.726180,0.031668,0.348843,0.790164,0.428001,0.555247,0.961261


In [26]:
surprise_df.sort_values(by='F-measure', ascending=False) 

,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
9,KNNBasic,0.945949,0.726180,0.031668,0.348843,0.790164,0.428001,0.555247,0.961261
4,KNNBaseline,0.870594,0.666896,0.052759,0.494724,0.806011,0.416210,0.548951,0.959268
1,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
6,SlopeOne,0.897614,0.687702,1.215158,1.542752,0.808579,0.406097,0.540657,0.959408
8,SVD,0.874252,0.670995,0.277401,0.030175,0.820219,0.402056,0.539607,0.960079
7,SVDpp,0.863433,0.661943,18.381992,3.228610,0.832268,0.398290,0.538754,0.962390
2,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
3,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573
5,NMF,0.918784,0.702863,0.483368,0.024606,0.786721,0.393843,0.524909,0.955988
0,CoClustering,0.940477,0.729357,0.443287,0.024041,0.792240,0.385911,0.519007,0.954984


In [27]:
surprise_df.sort_values(by='NDCG', ascending=False)

,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
7,SVDpp,0.863433,0.661943,18.381992,3.228610,0.832268,0.398290,0.538754,0.962390
9,KNNBasic,0.945949,0.726180,0.031668,0.348843,0.790164,0.428001,0.555247,0.961261
1,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
8,SVD,0.874252,0.670995,0.277401,0.030175,0.820219,0.402056,0.539607,0.960079
6,SlopeOne,0.897614,0.687702,1.215158,1.542752,0.808579,0.406097,0.540657,0.959408
4,KNNBaseline,0.870594,0.666896,0.052759,0.494724,0.806011,0.416210,0.548951,0.959268
2,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
3,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573
5,NMF,0.918784,0.702863,0.483368,0.024606,0.786721,0.393843,0.524909,0.955988
0,CoClustering,0.940477,0.729357,0.443287,0.024041,0.792240,0.385911,0.519007,0.954984


In [28]:
sim_options = {'name': 'pearson_baseline',
               'user_based': False  # compute  similarities between items
               }
algo = KNNBaseline(sim_options=sim_options)

results = recommendation(algo,trainset,testset)
df = pd.DataFrame([['KNNBaseline (pearson_baseline)', results[0], results[1], results[2], results[3], results[4], results[5], results[6], results[7]]], columns= ['Algorithm', 'test_rmse', 'test_mae', 'fit_time', 'test_time', 'Precision', 'Recall', 'F-measure', 'NDCG'])
surprise_df = pd.concat([df, surprise_df], ignore_index=True)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
RMSE: 0.8511
MAE:  0.6493


In [29]:
surprise_df.head()

,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
0,KNNBaseline (pearson_baseline),0.851100,0.649310,2.053726,1.752640,0.841038,0.413989,0.554858,0.964526
1,CoClustering,0.940477,0.729357,0.443287,0.024041,0.792240,0.385911,0.519007,0.954984
2,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
3,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
4,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573


In [30]:
surprise_df.to_csv('Surprise_results.csv')

In [31]:
surprise_df.sort_values(by='test_rmse') 

,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
0,KNNBaseline (pearson_baseline),0.851100,0.649310,2.053726,1.752640,0.841038,0.413989,0.554858,0.964526
8,SVDpp,0.863433,0.661943,18.381992,3.228610,0.832268,0.398290,0.538754,0.962390
2,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
5,KNNBaseline,0.870594,0.666896,0.052759,0.494724,0.806011,0.416210,0.548951,0.959268
9,SVD,0.874252,0.670995,0.277401,0.030175,0.820219,0.402056,0.539607,0.960079
3,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
4,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573
7,SlopeOne,0.897614,0.687702,1.215158,1.542752,0.808579,0.406097,0.540657,0.959408
6,NMF,0.918784,0.702863,0.483368,0.024606,0.786721,0.393843,0.524909,0.955988
1,CoClustering,0.940477,0.729357,0.443287,0.024041,0.792240,0.385911,0.519007,0.954984


In [32]:
surprise_df.sort_values(by='F-measure', ascending=False) 

,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
10,KNNBasic,0.945949,0.726180,0.031668,0.348843,0.790164,0.428001,0.555247,0.961261
0,KNNBaseline (pearson_baseline),0.851100,0.649310,2.053726,1.752640,0.841038,0.413989,0.554858,0.964526
5,KNNBaseline,0.870594,0.666896,0.052759,0.494724,0.806011,0.416210,0.548951,0.959268
2,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
7,SlopeOne,0.897614,0.687702,1.215158,1.542752,0.808579,0.406097,0.540657,0.959408
9,SVD,0.874252,0.670995,0.277401,0.030175,0.820219,0.402056,0.539607,0.960079
8,SVDpp,0.863433,0.661943,18.381992,3.228610,0.832268,0.398290,0.538754,0.962390
3,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
4,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573
6,NMF,0.918784,0.702863,0.483368,0.024606,0.786721,0.393843,0.524909,0.955988


In [33]:
surprise_df.sort_values(by='NDCG', ascending=False)

,Algorithm,test_rmse,test_mae,fit_time,test_time,Precision,Recall,F-measure,NDCG
0,KNNBaseline (pearson_baseline),0.851100,0.649310,2.053726,1.752640,0.841038,0.413989,0.554858,0.964526
8,SVDpp,0.863433,0.661943,18.381992,3.228610,0.832268,0.398290,0.538754,0.962390
10,KNNBasic,0.945949,0.726180,0.031668,0.348843,0.790164,0.428001,0.555247,0.961261
2,BaselineOnly,0.870236,0.671585,0.027568,0.016948,0.831694,0.409534,0.548822,0.960277
9,SVD,0.874252,0.670995,0.277401,0.030175,0.820219,0.402056,0.539607,0.960079
7,SlopeOne,0.897614,0.687702,1.215158,1.542752,0.808579,0.406097,0.540657,0.959408
5,KNNBaseline,0.870594,0.666896,0.052759,0.494724,0.806011,0.416210,0.548951,0.959268
3,KNNWithZScore,0.890163,0.677184,0.042391,0.402292,0.806120,0.402460,0.536880,0.959076
4,KNNWithMeans,0.891456,0.683259,0.029298,0.365023,0.804918,0.391705,0.526967,0.958573
6,NMF,0.918784,0.702863,0.483368,0.024606,0.786721,0.393843,0.524909,0.955988
